In [50]:
from geneticengine.grammar.decorators import abstract
from geneticengine.grammar.grammar import extract_grammar
from geneticengine.algorithms.gp.gp import GeneticProgramming
from geneticengine.random.sources import NativeRandomSource
from geneticengine.representations.tree.initializations import MaxDepthDecider
from geneticengine.representations.tree.treebased import TreeBasedRepresentation
from geneticengine.problems import SingleObjectiveProblem
from geneticengine.evaluation.budget import EvaluationBudget, TimeBudget
from geneticengine.grammar.metahandlers.ints import IntRange
from geneticengine.evaluation.recorder import CSVSearchRecorder
from geneticengine.evaluation.tracker import ProgressTracker
from geneticengine.problems import MultiObjectiveProblem

from analysis_helpers import calculate_expression_complexity

import time

from abc import ABC
from dataclasses import dataclass

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils import resample
from sklearn import metrics

from typing import Annotated
import pandas as pd

In [51]:
dataset = pd.read_csv('../../datasets/parkinsons.csv')
X = dataset.drop(columns=['name', 'status']).values
y = dataset['status'].values

In [52]:
n_features = X.shape[1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [53]:
weight_config = {
    "w_operator" : 0.3,
    "w_column": 0.2,
    "w_depth": 0.5
}

In [54]:
#core classes
class Feature(ABC):
    def evaluate(self, X):
        pass

@dataclass
class PrimitiveFeature(Feature):
    column_index: Annotated[int, IntRange(0,n_features-1)]

    def evaluate(self, X):
        return X[:, self.column_index]
    
    def __str__(self):
        return f"column_{dataset.columns[self.column_index+1]}"
        
@dataclass
class Add(Feature):
    left : Feature #type feature
    right : Feature #type feature
    def evaluate(self, X):
        return self.left.evaluate(X) + self.right.evaluate(X)
    def __str__(self):
        return f"({self.left} + {self.right})"

@dataclass
class Subtract(Feature):
    left : Feature
    right : Feature
    def evaluate(self, X):
        return self.left.evaluate(X) - self.right.evaluate(X)
    def __str__(self):
        return f"({self.left} - {self.right})"


In [55]:
#grammar
components = [
    PrimitiveFeature,
    Add,
    Subtract
]

grammar = extract_grammar(components, Feature)
print(grammar)

Grammar<Starting=Feature,Productions={
Feature -> PrimitiveFeature(column_index: Annotated[int])|
	Add(left: Feature, right: Feature)|
	Subtract(left: Feature, right: Feature)
}


In [56]:
#fitness function
def fitness_function(feature: Feature) -> float:
    start = time.perf_counter()

    feature_values = feature.evaluate(X_train)

    X_combined = np.hstack([X_train, feature_values.reshape(-1,1)])
    
    clf = make_pipeline(StandardScaler(), LogisticRegression(random_state=0, max_iter=200, solver='liblinear'))
    f1 = cross_val_score(clf, X_combined, y_train, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), scoring='f1_macro').mean()

    complexity = calculate_expression_complexity(str(feature), weight_config)

    elapsed = time.perf_counter() - start

    return [f1, complexity, elapsed]


In [57]:
#GP setup
rnd = NativeRandomSource(123)
decider = MaxDepthDecider(rnd, grammar, max_depth=5)
representation = TreeBasedRepresentation(grammar, decider)
objective = MultiObjectiveProblem(fitness_function=fitness_function, minimize=[False, True, True])

gp = GeneticProgramming(
    problem=objective,
    budget=TimeBudget(30),
    representation=representation,
    random=rnd,
    tracker= ProgressTracker(
        objective,
        recorders=[CSVSearchRecorder(csv_path='../../gp_outputs/tests.csv', problem=objective, fields={"Eval Time": lambda t,i,p:i.get_fitness(p).fitness_components[2],
                                                                                                       "Expression": lambda t, i, p: i.get_phenotype(),
                                                                                                       "F1 Score": lambda t, i, p: i.get_fitness(p).fitness_components[0],
                                                                                                       "Complexity": lambda t, i, p: i.get_fitness(p).fitness_components[1],
                                                                                                       }, only_record_best_individuals=True)]
    ),
    population_size=50,
)

solutions = gp.search()


In [58]:
best = max(solutions, key=lambda row: row.get_fitness(objective).fitness_components[0])

In [59]:
print(best.get_fitness(objective).fitness_components)

[np.float64(0.8161973121039967), 1.5, 0.017959199991310015]


In [60]:
# print(f"{alg.get_fitness(gp.get_problem())}\n{alg.get_phenotype()}") #f1, expression complexity, evaluation time

In [61]:


# Add this cell to check baseline
clf = make_pipeline(StandardScaler(), LogisticRegression(random_state=0, max_iter=200, solver='liblinear'))
baseline_f1 = cross_val_score(clf, X_train, y_train, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), scoring='f1_macro').mean()
print(f"Baseline F1 Score (original features only): {baseline_f1}")
print(f"Your GP F1 Score: {best.get_fitness(objective).fitness_components[0]}")
print(f"Improvement: {best.get_fitness(objective).fitness_components[0] - baseline_f1}")


Baseline F1 Score (original features only): 0.7880184180695691
Your GP F1 Score: 0.8161973121039967
Improvement: 0.028178894034427637
